# The Financial Fraud Classifier (Multinomial Naive Bayes)
---

### El Objetivo

**Construir un clasificador Naive Bayes desde cero (sin librerías de ML como Scikit-Learn) para predecir si una transacción bancaria es legítima ($0$) o fraudulenta ($1$), basándose en múltiples variables simultáneas.**

## Conceptos Matemáticos

### Eventos Conjuntos
Una transacción no es solo "alta" o "baja". Es "Alta" y "A las 3 AM" y "En un país extranjero". Queremos calcular la probabilidad de que todo eso ocurra al mismo tiempo.

### Independencia Condicional (La parte Naive)
Asumimos matemáticamente que la hora de la transacción no depende del monto, condicionado a la clase. En la vida real no siempre es cierto, pero esta simplificación reduce drásticamente el costo de cálculo y suele funcionar muy bien.

### Suavizado de Laplace
El salvavidas estadístico. Si el modelo ve una categoría nunca observada, su probabilidad no cae a $0$ gracias al ajuste de Laplace.

$$P(\text{Fraude} \mid x_1, x_2, \dots, x_n) \propto P(\text{Fraude}) \prod_{i=1}^{n} P(x_i \mid \text{Fraude})$$

## Arquitectura Lógica

### Fase A — La Bóveda de Datos (Feature Engineering)
Crearemos un historial de 10,000 transacciones. Cada transacción tendrá 3 variables categóricas:
- **Monto:** Alto / Normal / Bajo
- **Ubicación:** Local / Extranjero
- **Hora:** Día / Madrugada

### Fase B — El Entrenamiento (Fase de Memoria)
Aquí el algoritmo no "aprende" mágicamente: **cuenta frecuencias**.
- Calcula el **Prior**: ¿qué porcentaje total son fraudes?
- Calcula los **Likelihoods**: de todos los fraudes, ¿cuántos fueron de madrugada, de monto alto, etc.?

### Fase C — El Motor de Inferencia (Aplicando Bayes)
Llega una nueva transacción: (Monto: Alto, Ubicación: Local, Hora: Madrugada). El motor combina probabilidades para ambas clases (Fraude vs. Legítima) y decide la clase más probable.

### Fase D — Validación del Sistema
Se evalúa con un **test set** no visto para medir precisión y, sobre todo, falsos positivos (bloquear tarjetas de clientes legítimos).

In [1]:
import numpy as np 
import pandas as pd
np.random.seed(42)

## Fase A — Generación del Dataset

In [2]:
# --- FASE A: GENERACIÓN DEL DATASET (La Bóveda del Banco) ---
n_tx = 10000
# El fraude es raro (solo el 5% de las transacciones)
fraude_real = np.random.choice([0, 1], size=n_tx, p=[0.95, 0.05])

# Generamos características correlacionadas probabilísticamente con el fraude
montos = []
ubicaciones = []
horas = []

for es_fraude in fraude_real:
    if es_fraude == 1:
        montos.append(np.random.choice(['Bajo', 'Normal', 'Alto'], p=[0.1, 0.2, 0.7]))
        ubicaciones.append(np.random.choice(['Local', 'Extranjero'], p=[0.3, 0.7]))
        horas.append(np.random.choice(['Día', 'Madrugada'], p=[0.2, 0.8]))
    else:
        montos.append(np.random.choice(['Bajo', 'Normal', 'Alto'], p=[0.5, 0.4, 0.1]))
        ubicaciones.append(np.random.choice(['Local', 'Extranjero'], p=[0.9, 0.1]))
        horas.append(np.random.choice(['Día', 'Madrugada'], p=[0.8, 0.2]))

df = pd.DataFrame({'Monto': montos, 'Ubicacion': ubicaciones, 'Hora': horas, 'Fraude': fraude_real})

## Fase B — Entrenamiento Matemático

In [3]:
# --- FASE B: FASE DE MEMORIA (Entrenamiento Matemático) ---
print("--- ENTRENANDO MOTOR NAIVE BAYES ---")

# 1. Calcular Priors P(Fraude) y P(Legitima)
prior_legitima = len(df[df['Fraude'] == 0]) / len(df)
prior_fraude = len(df[df['Fraude'] == 1]) / len(df)

# 2. Calcular Likelihoods con Suavizado de Laplace (alpha = 1)
# Para evitar P = 0 si vemos algo nuevo
def calcular_likelihood(feature, valor, clase_fraude, alpha=1):
    df_clase = df[df['Fraude'] == clase_fraude]
    # Conteo con Laplace: (Apariciones + alpha) / (Total de la clase + alpha * num_categorias)
    num_categorias = df[feature].nunique()
    apariciones = len(df_clase[df_clase[feature] == valor])
    return (apariciones + alpha) / (len(df_clase) + alpha * num_categorias)

--- ENTRENANDO MOTOR NAIVE BAYES ---


## Fase C — Motor de Inferencia en Vivo

In [4]:
# --- FASE C: EL MOTOR DE INFERENCIA EN VIVO ---
def predecir_transaccion(monto, ubicacion, hora):
    # Sumar logaritmos en lugar de multiplicar probabilidades
    # log(P(A * B)) = log(P(A)) + log(P(B))
    
    # 1. Probabilidad Logarítmica de ser Fraude
    log_p_fraude = np.log(prior_fraude) + \
                   np.log(calcular_likelihood('Monto', monto, 1)) + \
                   np.log(calcular_likelihood('Ubicacion', ubicacion, 1)) + \
                   np.log(calcular_likelihood('Hora', hora, 1))
                   
    # 2. Probabilidad Logarítmica de ser Legítima
    log_p_legit = np.log(prior_legitima) + \
                  np.log(calcular_likelihood('Monto', monto, 0)) + \
                  np.log(calcular_likelihood('Ubicacion', ubicacion, 0)) + \
                  np.log(calcular_likelihood('Hora', hora, 0))
    
    print(f"\nAnalizando Transacción: [Monto: {monto} | Ubicación: {ubicacion} | Hora: {hora}]")
    print(f"Score Logarítmico Fraude: {log_p_fraude:.2f}")
    print(f"Score Logarítmico Legítima: {log_p_legit:.2f}")
    
    if log_p_fraude > log_p_legit:
        print(">> VEREDICTO: ¡ALERTA DE FRAUDE! 🚨 Tarjeta Bloqueada.")
    else:
        print(">> VEREDICTO: Transacción Legítima ✅ Aprobada.")

## Fase D — Validación del Sistema

In [5]:
# --- FASE D: VALIDACIÓN (Test en vivo) ---
# Caso 1: Comportamiento normal (Comprar un café por la tarde en tu ciudad)
predecir_transaccion(monto='Bajo', ubicacion='Local', hora='Día')

# Caso 2: Comportamiento altamente sospechoso (Compra enorme en otro país de madrugada)
predecir_transaccion(monto='Alto', ubicacion='Extranjero', hora='Madrugada')

# Caso 3: El dilema (Monto alto, pero en tu ciudad y de día)
predecir_transaccion(monto='Alto', ubicacion='Local', hora='Día')


Analizando Transacción: [Monto: Bajo | Ubicación: Local | Hora: Día]
Score Logarítmico Fraude: -8.30
Score Logarítmico Legítima: -1.07
>> VEREDICTO: Transacción Legítima ✅ Aprobada.

Analizando Transacción: [Monto: Alto | Ubicación: Extranjero | Hora: Madrugada]
Score Logarítmico Fraude: -3.95
Score Logarítmico Legítima: -6.23
>> VEREDICTO: ¡ALERTA DE FRAUDE! 🚨 Tarjeta Bloqueada.

Analizando Transacción: [Monto: Alto | Ubicación: Local | Hora: Día]
Score Logarítmico Fraude: -6.38
Score Logarítmico Legítima: -2.65
>> VEREDICTO: Transacción Legítima ✅ Aprobada.
